<a href="https://colab.research.google.com/github/Minh-Quan-hub/build-ai-agents-and-chatbots-with-langgraph-2021112/blob/main/Agentic_AI_in_Customer_Service_and_Sales.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### **1. Install dependencies**



In [1]:
%pip install -qU langchain-openai langchain-core langgraph pydantic

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 133.0/133.0 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 146.4/146.4 kB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 573.8/573.8 kB 20.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 260.4/260.4 kB 16.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 488.1/488.1 kB 25.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 53.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.1/58.1 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 164.6/164.6 kB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 869.0/869.0 kB 36.0 MB/s eta 0:00:00


### **2. Configure the OpenAI API key**

In [10]:
import os
from google.colab import userdata

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

### **3. Imports, data, and system prompts**

In [5]:
from typing import Annotated, Literal

from pydantic import BaseModel, Field
from langchain_core.messages import AIMessage, AnyMessage, HumanMessage, SystemMessage
from langchain_openai import ChatOpenAI
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import END, START, MessagesState, StateGraph


PRODUCT_CATALOG = """
- FocusBook Air: $899, 13-inch, 16 GB RAM, 512 GB SSD. Best for students,
  writing, browsing, and everyday productivity. Battery up to 15 hours.
- CreatorBook Pro: $1,499, 15-inch, 32 GB RAM, 1 TB SSD. Best for video
  editing, programming, and demanding creative work. Battery up to 11 hours.
- TravelBook Flex: $1,099, 14-inch 2-in-1, 16 GB RAM, 512 GB SSD. Best for
  frequent travelers who want a light laptop and tablet mode. Battery up to
  13 hours.
"""

ORDER_DATA = """
- ORD-1001: CreatorBook Pro; status = shipped; estimated delivery = October 20;
  quantity = 1.
- ORD-1002: FocusBook Air; status = processing; estimated delivery = October 23;
  quantity = 1.
"""

SYSTEM_PROMPT = """
You are Nova, a helpful customer-service and sales assistant for NovaTech
laptops. Be accurate, concise, friendly, and transparent. Use the conversation
history to resolve references such as 'that laptop' or 'my order'. Never invent
an order, product specification, delivery date, or policy. If a requested detail
is not in the supplied business information, say that you do not have it and
ask for the missing detail. You are not authorized to issue refunds or change
orders; explain the next support step instead.
"""

PRODUCT_PROMPT = SYSTEM_PROMPT + f"""

You are the Product Recommendation Agent. Recommend one or two products based
on the customer's needs, budget, and priorities. Briefly explain the match and
mention a trade-off when useful. Use only this catalog:
{PRODUCT_CATALOG}
"""

REFUND_PROMPT = SYSTEM_PROMPT + """

You are the Refund Policy Agent. Explain the policy in plain language:
- Customers may request a return within 30 days of delivery.
- Items must be in good condition with the original accessories.
- Approved refunds are sent to the original payment method within 5-7 business
  days after the returned item is inspected.
- Open a support request with the order number to begin.
Do not promise approval before inspection.
"""

ORDER_PROMPT = SYSTEM_PROMPT + f"""

You are the Order Status Agent. Look up the order number in this internal
read-only order data and report the status, item, quantity, and estimated
delivery when available:
{ORDER_DATA}
If the order number is missing or not listed, ask the customer to verify it.
Never claim that an order was changed.
"""

### **4. Define graph state and the routing decision**

In [6]:
class SupportState(MessagesState):
    """Messages plus the route selected for the current turn."""

    route: str


class RouteDecision(BaseModel):
    route: Literal["PRODUCT", "REFUND", "ORDER"] = Field(
        description="The specialist that should answer the latest user request."
    )


model = ChatOpenAI(model="gpt-4o-mini", temperature=0)
router_model = model.with_structured_output(RouteDecision)


def route_request(state: SupportState):
    """Classify only the latest request while using history for context."""
    router_prompt = SYSTEM_PROMPT + """

You are the routing agent. Read the full conversation, especially the latest
user message, and select exactly one specialist:
- PRODUCT: product information, comparisons, or recommendations
- REFUND: returns, refunds, exchanges, or return eligibility
- ORDER: order status, delivery, order details, or order lookup
"""
    decision = router_model.invoke(
        [SystemMessage(content=router_prompt), *state["messages"]]
    )
    return {"route": decision.route}


def specialist_node(system_prompt: str, agent_name: str):
    """Create an independent specialist node with its own instructions."""
    def node(state: SupportState):
        response = model.invoke(
            [SystemMessage(content=system_prompt), *state["messages"]]
        )
        # Keep the agent identity visible for testing/debugging without adding
        # another user-visible message to the conversation.
        response.name = agent_name
        return {"messages": [response]}

    return node


product_agent = specialist_node(PRODUCT_PROMPT, "ProductRecommendationAgent")
refund_agent = specialist_node(REFUND_PROMPT, "RefundPolicyAgent")
order_agent = specialist_node(ORDER_PROMPT, "OrderStatusAgent")

### **5. Build the routed LangGraph with MemorySaver**

In [7]:
builder = StateGraph(SupportState)
builder.add_node("router", route_request)
builder.add_node("product_agent", product_agent)
builder.add_node("refund_agent", refund_agent)
builder.add_node("order_agent", order_agent)

builder.add_edge(START, "router")
builder.add_conditional_edges(
    "router",
    lambda state: state["route"],
    {
        "PRODUCT": "product_agent",
        "REFUND": "refund_agent",
        "ORDER": "order_agent",
    },
)
builder.add_edge("product_agent", END)
builder.add_edge("refund_agent", END)
builder.add_edge("order_agent", END)

# MemorySaver stores graph state by thread_id for this notebook session.
memory = MemorySaver()
chatbot = builder.compile(checkpointer=memory)


def ask(user_text: str, thread_id: str = "demo-customer-1") -> str:
    """Send one message and return the selected agent's response."""
    result = chatbot.invoke(
        {"messages": [HumanMessage(content=user_text)]},
        config={"configurable": {"thread_id": thread_id}},
    )
    print(f"[{result['route']}] {result['messages'][-1].content}")
    return result["messages"][-1].content

### **6. Three-turn test: three different agents, one remembered thread**

In [8]:
thread_id = "rubric-demo-1"

ask("I need a laptop for video editing and programming. Which one do you recommend?", thread_id)
ask("What is your refund policy if I change my mind about that laptop?", thread_id)
ask("Can you check the delivery status of order ORD-1001?", thread_id)

# Verify that the same thread contains at least three user turns and three
# assistant responses. This is evidence that MemorySaver is being used.
saved_state = chatbot.get_state({"configurable": {"thread_id": thread_id}})
saved_messages = saved_state.values["messages"]
user_turns = sum(isinstance(message, HumanMessage) for message in saved_messages)
assistant_turns = sum(isinstance(message, AIMessage) for message in saved_messages)
assert user_turns >= 3 and assistant_turns >= 3
print(f"\nMemorySaver verification: {user_turns} user turns and {assistant_turns} assistant turns saved.")

[PRODUCT] For video editing and programming, I recommend the **CreatorBook Pro**. It features a 15-inch display, 32 GB of RAM, and a 1 TB SSD, making it ideal for demanding creative work and multitasking. 

The trade-off is that it has a shorter battery life of up to 11 hours compared to other models, but its performance capabilities are well-suited for your needs. If you have any specific budget constraints or preferences, let me know!
[REFUND] Our refund policy allows you to request a return within 30 days of delivery. Here are the key points:

- The laptop must be in good condition and include all original accessories.
- If your return is approved, the refund will be processed to your original payment method within 5-7 business days after we inspect the returned item.
- To begin the process, you'll need to open a support request and provide your order number.

Please note that I can't promise approval before the inspection. If you need help with the support request, let me know!
[OR

### **7. Interactive chat loop**

In [9]:
import uuid
live_thread_id = str(uuid.uuid4())
while True:
     user_text = input("You: ").strip()
     if user_text.lower() in {"quit", "exit"}:
         print("Nova: Thanks for chatting. Goodbye!")
         break
     ask(user_text, live_thread_id)

You: I need a laptop for video editing and programming. Which one do you recommend?
[PRODUCT] For video editing and programming, I recommend the **CreatorBook Pro**. It features a 15-inch display, 32 GB of RAM, and a 1 TB SSD, making it ideal for demanding creative work and multitasking. 

The trade-off is that it has a shorter battery life of up to 11 hours compared to other models, but its performance capabilities are well-suited for your needs. If you have any specific budget constraints or additional requirements, let me know!
You: What is the refund policy if I change my mind about that laptop?
[REFUND] Our refund policy allows you to request a return within 30 days of delivery. Here are the key points:

- The laptop must be in good condition and include all original accessories.
- If your return is approved, the refund will be processed to your original payment method within 5-7 business days after we inspect the returned item.
- To start the process, you'll need to open a suppor